In [7]:
# Installation automatique des dépendances requises dans le noyau Jupyter actuel
%pip install -r ../requirements.txt

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


# 🔎 Étape 3 : Analyse Exploratoire des Données (EDA) (Squelette Étudiant)

Cette étape correspond au troisième chapitre du cours. L'objectif est d'explorer et de résumer les propriétés statistiques fondamentales de vos données et de réaliser du **Feature Engineering** pour enrichir vos modèles.

### 1. Préparation de l'environnement

In [8]:
import os
import sys
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath('..'))
from src import data_clean as dc

print("Librairies importées pour l'EDA !")

Librairies importées pour l'EDA !


### 2. Chargement des données nettoyées

In [9]:
df = pd.read_csv("../data/processed/ai_student_cleaned.csv")

# Pas de colonne temporelle dans ce dataset : on saute pd.to_datetime().
print(f"Dimensions : {df.shape}")
df.head()

Dimensions : (50000, 20)


,Student_ID,Major_Category,Year_of_Study,Pre_Semester_GPA,Weekly_GenAI_Hours,Primary_Use_Case,Prompt_Engineering_Skill,Tool_Diversity,Paid_Subscription,Traditional_Study_Hours,Perceived_AI_Dependency,Institutional_Policy,Anxiety_Level_During_Exams,Post_Semester_GPA,Skill_Retention_Score,Burnout_Risk_Level,Secteur_IA_Adoption_Rate,Avg_Salary_Index,Recommended_Study_Hours_Week,High_Stakes_Exams
0,100001,Humanities,Senior,2.418,23.31,Copywriting/Drafting,Beginner,1,True,8.13,5,Allowed_With_Citation,6,2.393,86.44,High,0.52,0.9,10,False
1,100002,Medical,Junior,3.821,1.12,Ideation,Advanced,5,False,16.65,3,Allowed_With_Citation,9,3.696,69.39,Low,0.61,1.5,18,True
2,100003,Business,Freshman,3.398,21.26,Summarizing_Reading,Beginner,2,False,10.35,5,Strict_Ban,9,3.499,73.93,Medium,0.74,1.2,12,False
3,100004,Business,Senior,3.789,1.82,Copywriting/Drafting,Intermediate,4,False,15.23,2,Allowed_With_Citation,2,4.000,63.58,Medium,0.74,1.2,12,False
4,100005,STEM,Sophomore,3.635,9.29,Debugging/Troubleshooting,Advanced,4,False,12.55,4,Allowed_With_Citation,4,3.798,100.00,Medium,0.87,1.4,15,True


### 3. Statistiques Descriptives

**À COMPLÉTER PAR L'ÉTUDIANT :**
Générez les résumés statistiques globaux et par groupes/catégories de votre jeu de données.

In [10]:
# Résumé statistique global des variables numériques
print("=== Statistiques descriptives globales ===")
print(df.describe().round(2))

# Agrégation par filière : moyenne du GPA, heures GenAI, rétention
print("\n=== Profil moyen par filière (Major_Category) ===")
by_major = df.groupby("Major_Category")[[
    "Pre_Semester_GPA", "Post_Semester_GPA",
    "Weekly_GenAI_Hours", "Traditional_Study_Hours",
    "Skill_Retention_Score", "Anxiety_Level_During_Exams",
]].mean().round(2)
print(by_major)

# Agrégation par politique institutionnelle : impact sur le GPA et l'anxiété
print("\n=== Effet de la politique institutionnelle ===")
by_policy = df.groupby("Institutional_Policy").agg(
    n_students=("Student_ID", "count"),
    gpa_post_mean=("Post_Semester_GPA", "mean"),
    genai_hours_mean=("Weekly_GenAI_Hours", "mean"),
    anxiety_mean=("Anxiety_Level_During_Exams", "mean"),
    burnout_high_rate=("Burnout_Risk_Level", lambda s: (s == "High").mean()),
).round(3)
print(by_policy)

=== Statistiques descriptives globales ===
       Student_ID  Pre_Semester_GPA  Weekly_GenAI_Hours  Tool_Diversity  \
count    50000.00          50000.00            50000.00        50000.00   
mean    125000.50              3.15                8.43            2.80   
std      14433.90              0.48                8.27            1.19   
min     100001.00              1.18                0.00            1.00   
25%     112500.75              2.83                2.39            2.00   
50%     125000.50              3.21                5.80            3.00   
75%     137500.25              3.52               11.72            4.00   
max     150000.00              4.00               40.00            5.00   

       Traditional_Study_Hours  Perceived_AI_Dependency  \
count                 50000.00                 50000.00   
mean                     11.21                     3.51   
std                       5.16                     1.82   
min                       1.00               

### 4. Ingénierie de variables (Feature Engineering)

**À COMPLÉTER PAR L'ÉTUDIANT :**
Appliquez la fonction `feature_engineering` de `src.data_clean` pour extraire des indicateurs temporels de base, et ajoutez d'autres variables dérivées complexes adaptées à votre problématique.

In [11]:
# Application de la feature_engineering définie dans src/data_clean.py
df_feat = dc.feature_engineering(df)

# Colonnes ajoutées par la fonction
new_cols = [c for c in df_feat.columns if c not in df.columns]
print("Variables dérivées créées :", new_cols)
df_feat[new_cols + ["Pre_Semester_GPA", "Weekly_GenAI_Hours"]].head()

Variables dérivées créées : ['AI_Usage_Level', 'GPA_Squared']


,AI_Usage_Level,GPA_Squared,Pre_Semester_GPA,Weekly_GenAI_Hours
0,1,5.846724,2.418,23.31
1,0,14.600041,3.821,1.12
2,1,11.546404,3.398,21.26
3,0,14.356521,3.789,1.82
4,0,13.213225,3.635,9.29


### 5. Analyse des Corrélations

**À COMPLÉTER PAR L'ÉTUDIANT :**
Analysez la matrice des corrélations des caractéristiques numériques à l'aide de Pandas.

In [12]:
# Matrice de corrélation de Pearson sur les variables numériques clés
cols = [
    "Pre_Semester_GPA", "Post_Semester_GPA", "GPA_Squared",
    "Weekly_GenAI_Hours", "AI_Usage_Level",
    "Traditional_Study_Hours", "Tool_Diversity",
    "Perceived_AI_Dependency", "Anxiety_Level_During_Exams",
    "Skill_Retention_Score",
]
correlations = df_feat[cols].corr(method="pearson").round(2)
print("=== Matrice de corrélation de Pearson ===")
print(correlations)

# Top 5 corrélations les plus fortes avec la cible Post_Semester_GPA
print("\n=== Top corrélations avec Post_Semester_GPA ===")
target_corr = correlations["Post_Semester_GPA"].drop("Post_Semester_GPA").abs().sort_values(ascending=False)
print(target_corr.head(5))

=== Matrice de corrélation de Pearson ===
                            Pre_Semester_GPA  Post_Semester_GPA  GPA_Squared  \
Pre_Semester_GPA                        1.00               0.93         1.00   
Post_Semester_GPA                       0.93               1.00         0.92   
GPA_Squared                             1.00               0.92         1.00   
Weekly_GenAI_Hours                     -0.00              -0.02        -0.00   
AI_Usage_Level                         -0.00              -0.01        -0.00   
Traditional_Study_Hours                -0.00               0.14        -0.00   
Tool_Diversity                         -0.01               0.03        -0.01   
Perceived_AI_Dependency                 0.00              -0.01         0.00   
Anxiety_Level_During_Exams             -0.00              -0.02        -0.00   
Skill_Retention_Score                   0.10               0.17         0.10   

                            Weekly_GenAI_Hours  AI_Usage_Level  \
Pre_Semeste